# preborn_derive — Phase B deterministic re-derivation

Runs AFTER both generators have been assembled to their working staging
schemas (`8_dev.synth_preborn_m.stg_*`, `8_dev.synth_preborn_i.stg_*`). Reads
ONLY synthetic data + the real vocabulary (not PII) — no raw MSDS/nnu, no DP
mechanism. Ports PREBORN's own CDM logic (`sp1_msds_omop/.../preborn/_common.py`)
onto the synthetic base, so every structural invariant holds by construction.

Steps (order matters):
1. **Unify base** — mother stg_* + infant stg_* → `{FINAL}.synth_*`, offsetting
   infant surrogate ids into a disjoint band, stamping site, setting person_role.
2. **Size calibration** — deterministic per-role downsample of the DP-modelled
   fact tables to the REAL per-role counts (TARGET_BY_ROLE). Downsample-only:
   never fabricates rows; drops synthetic rows so it needs no DP budget.
3. **infant → pregnancy assignment** — assign each infant to a pregnancy (liveborn
   slots); derive `synth_infant` with infant_id == person_id.
4. **infant visits + visit_detail** — designate nnu (Inpatient) infant visits and
   mint one visit_detail per nnu visit (PVID rule); every detail resolves to a parent.
5. **link** — FIRST visit_occurrence → pregnancy, THEN facts → their person's visit
   (inheriting the now-populated visit.pregnancy_id).
6. **episode / episode_event** — one episode per pregnancy; link 5 fact tables via
   episode.pregnancy_id = fact.pregnancy_id.
7. **observation_period** — one per person spanning its events.
8. **fact_relationship** — mother/child/twin/sibling person↔person pairs.
9. **cdm_source** — one row describing the SYNTHETIC release.
10. **used-vocabulary** — bundled vocab pruned to the synthetic used concept set.
11. **cleanup** — drop the _infant_map/_preg_slots/_nnu_visits helper tables.

In [0]:
%run ./preborn_config

In [0]:
VOCAB_SRC = "4_prod.omop"                 # real vocabulary source (NOT PII)
INFANT_ID_OFFSET = 500_000_000_000        # disjoint band for infant surrogate ids
MSTG = f"{WORKING_CATALOG}.{MOTHER_SCHEMA}"
ISTG = f"{WORKING_CATALOG}.{INFANT_SCHEMA}"
FINAL = f"{FINAL_CATALOG}.{FINAL_SCHEMA}"
FACTS = ["visit_occurrence", "measurement", "observation",
         "condition_occurrence", "procedure_occurrence"]

# mid(): deterministic sha2->BIGINT id, namespaced by grain. Verbatim from _common.py:7-13.
def _norm_sql(x):
    return f"coalesce(nullif(trim(cast({x} as string)), ''), '~')"

def mid(grain, site, *part_sqls):
    site_lit = "'" + str(site).replace("'", "''") + "'"
    inner = ", ".join([f"'{grain}'", _norm_sql(site_lit)] + [_norm_sql(p) for p in part_sqls])
    return f"conv(substr(sha2(concat_ws('|', {inner}), 256), 1, 15), 16, 10)"

def M(grain, *p):
    return mid(grain, SITE_ID, *p)

def _full_cols(tbl):
    return [f.name for f in spark.table(f"{SRC}.{tbl}").schema.fields]

def _sqltype(tbl, col):
    for f in spark.table(f"{SRC}.{tbl}").schema.fields:
        if f.name == col:
            return f.dataType.simpleString().upper()
    return "STRING"

def _stg_or_none(stg, t):
    try:
        spark.table(f"{stg}.stg_{t}")
        return f"{stg}.stg_{t}"
    except Exception:
        return None

def _offset_expr(col, off):
    return f"CASE WHEN `{col}` IS NULL THEN NULL ELSE `{col}` + {off} END"


def derive_all():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FINAL}")
    _unify_base()
    _calibrate_sizes()
    _assign_infant_pregnancy()
    _infant_visits_and_detail()
    _link()
    _episode_and_events()
    _observation_period()
    _fact_relationship()
    _cdm_source()
    build_used_vocabulary_preborn(FINAL)
    _cleanup_helpers()
    _summary()


# ---------------------------------------------------------------------------
# 1. Unify mother + infant staged base into synth_* (offset infant ids; stamp site)
# ---------------------------------------------------------------------------
def _unify_base():
    pcols = _full_cols("person")
    def _person_proj(stg, role, off):
        parts = []
        stg_cols = set(f.name for f in spark.table(f"{stg}.stg_person").schema.fields)
        for c in pcols:
            if c == "person_id":
                parts.append(f"{_offset_expr('person_id', off)} AS person_id")
            elif c == "person_role":
                parts.append(f"'{role}' AS person_role")
            elif c == "site_id":
                parts.append(f"'{SITE_ID}' AS site_id")
            elif c in stg_cols:
                parts.append(f"`{c}`")
            else:
                parts.append(f"CAST(NULL AS {_sqltype('person', c)}) AS `{c}`")
        return f"SELECT {', '.join(parts)} FROM {stg}.stg_person"
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_person AS
      {_person_proj(MSTG, 'mother', 0)}
      UNION ALL
      {_person_proj(ISTG, 'infant', INFANT_ID_OFFSET)}""")
    print("synth_person:", spark.table(f"{FINAL}.synth_person").count())

    pg_cols = _full_cols("pregnancy")
    stg_pg = set(f.name for f in spark.table(f"{MSTG}.stg_pregnancy").schema.fields)
    pparts = [(f"'{SITE_ID}' AS site_id" if c == "site_id"
               else (f"`{c}`" if c in stg_pg else f"CAST(NULL AS {_sqltype('pregnancy', c)}) AS `{c}`"))
              for c in pg_cols]
    spark.sql(f"CREATE OR REPLACE TABLE {FINAL}.synth_pregnancy AS "
              f"SELECT {', '.join(pparts)} FROM {MSTG}.stg_pregnancy")
    print("synth_pregnancy:", spark.table(f"{FINAL}.synth_pregnancy").count())

    for t in FACTS:
        cols = _full_cols(t)
        pk = "visit_occurrence_id" if t == "visit_occurrence" else f"{t}_id"
        def _fact_proj(stg, off):
            src = _stg_or_none(stg, t)
            if src is None:
                return None
            have = set(f.name for f in spark.table(src).schema.fields)
            parts = []
            for c in cols:
                if c == pk:
                    parts.append(f"{_offset_expr(pk, off)} AS {pk}")
                elif c == "person_id":
                    parts.append(f"{_offset_expr('person_id', off)} AS person_id")
                elif c == "preceding_visit_occurrence_id" and c in have:
                    parts.append(f"{_offset_expr(c, off)} AS {c}")
                elif c == "site_id":
                    parts.append(f"'{SITE_ID}' AS site_id")
                elif c in ("pregnancy_id", "visit_occurrence_id", "visit_detail_id") and c != pk:
                    parts.append(f"CAST(NULL AS BIGINT) AS {c}")   # set in _link / _infant_visits
                elif c in have:
                    parts.append(f"`{c}`")
                else:
                    parts.append(f"CAST(NULL AS {_sqltype(t, c)}) AS `{c}`")
            return f"SELECT {', '.join(parts)} FROM {src}"
        m = _fact_proj(MSTG, 0)
        i = _fact_proj(ISTG, INFANT_ID_OFFSET)
        union = "\nUNION ALL\n".join([q for q in (m, i) if q])
        spark.sql(f"CREATE OR REPLACE TABLE {FINAL}.synth_{t} AS {union}")
        print(f"synth_{t}:", spark.table(f"{FINAL}.synth_{t}").count())


# ---------------------------------------------------------------------------
# 2. Size calibration — deterministic per-role downsample to real counts.
#    DP-modelled sequence lengths can overshoot; drop the excess synthetic rows
#    (hash-ranked, seedless-deterministic). Never upsamples. Runs BEFORE linking,
#    so visit deletions cannot orphan fact->visit links; dangling
#    preceding_visit_occurrence_id pointers are nulled afterwards.
# ---------------------------------------------------------------------------
def _calibrate_sizes():
    for t, targets in TARGET_BY_ROLE.items():
        pk = "visit_occurrence_id" if t == "visit_occurrence" else f"{t}_id"
        for role, tgt in targets.items():
            cur = spark.sql(f"""SELECT COUNT(*) c FROM {FINAL}.synth_{t} f
              JOIN {FINAL}.synth_person p ON p.person_id=f.person_id
              WHERE p.person_role='{role}'""").first()["c"]
            if cur <= tgt:
                print(f"calibrate {t}/{role}: {cur} <= target {tgt} — no downsample")
                continue
            spark.sql(f"""DELETE FROM {FINAL}.synth_{t} WHERE `{pk}` IN (
              SELECT `{pk}` FROM (
                SELECT f.`{pk}`,
                       ROW_NUMBER() OVER (ORDER BY abs(hash(f.`{pk}`)), f.`{pk}`) AS rn
                FROM {FINAL}.synth_{t} f
                JOIN {FINAL}.synth_person p ON p.person_id=f.person_id
                WHERE p.person_role='{role}'
              ) WHERE rn > {tgt})""")
            print(f"calibrate {t}/{role}: {cur} -> {tgt}")
    # visits were deleted -> null any dangling preceding pointers
    spark.sql(f"""MERGE INTO {FINAL}.synth_visit_occurrence v
      USING (
        SELECT v2.visit_occurrence_id FROM {FINAL}.synth_visit_occurrence v2
        WHERE v2.preceding_visit_occurrence_id IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {FINAL}.synth_visit_occurrence p
                          WHERE p.visit_occurrence_id = v2.preceding_visit_occurrence_id)
      ) d ON v.visit_occurrence_id = d.visit_occurrence_id
      WHEN MATCHED THEN UPDATE SET v.preceding_visit_occurrence_id = NULL""")
    print("calibration complete; dangling preceding_visit pointers nulled")


# ---------------------------------------------------------------------------
# 3. infant -> pregnancy assignment + synth_infant extension (infant_id == person_id)
# ---------------------------------------------------------------------------
def _assign_infant_pregnancy():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._preg_slots AS
      WITH pn AS (
        SELECT pregnancy_id, person_id,
               GREATEST(1, COALESCE(CAST(pregnancy_number_liveborn AS INT), 1)) AS n_slots
        FROM {FINAL}.synth_pregnancy
      )
      SELECT pregnancy_id, person_id AS mother_person_id,
             ROW_NUMBER() OVER (ORDER BY pregnancy_id) AS slot_rank
      FROM pn LATERAL VIEW EXPLODE(SEQUENCE(1, n_slots)) e AS k""")
    n_slots = spark.table(f"{FINAL}._preg_slots").count()
    assert n_slots > 0, "no pregnancy slots — mother generator produced no pregnancies"

    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._infant_map AS
      WITH inf AS (
        SELECT person_id AS infant_person_id,
               ROW_NUMBER() OVER (ORDER BY person_id) AS r
        FROM {FINAL}.synth_person WHERE person_role='infant'
      )
      SELECT i.infant_person_id, s.pregnancy_id, s.mother_person_id
      FROM inf i JOIN {FINAL}._preg_slots s
        ON s.slot_rank = pmod(i.r - 1, {n_slots}) + 1""")

    attr = _stg_or_none(ISTG, "infant_attr")
    icols = _full_cols("infant")
    attr_cols = set(f.name for f in spark.table(attr).schema.fields) if attr else set()
    parts = []
    for c in icols:
        if c == "pregnancy_id":
            parts.append("map.pregnancy_id AS pregnancy_id")
        elif c == "infant_id":
            parts.append("map.infant_person_id AS infant_id")
        elif c == "site_id":
            parts.append(f"'{SITE_ID}' AS site_id")
        elif c in attr_cols:
            parts.append(f"a.`{c}`")
        else:
            parts.append(f"CAST(NULL AS {_sqltype('infant', c)}) AS `{c}`")
    attr_join = (f"LEFT JOIN {attr} a ON a.person_id + {INFANT_ID_OFFSET} = map.infant_person_id"
                 if attr else "")
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_infant AS
      SELECT {', '.join(parts)} FROM {FINAL}._infant_map map {attr_join}""")
    bad = spark.sql(f"""SELECT COUNT(*) c FROM {FINAL}.synth_infant i
      WHERE NOT EXISTS (SELECT 1 FROM {FINAL}.synth_pregnancy pr WHERE pr.pregnancy_id=i.pregnancy_id)""").first()["c"]
    assert bad == 0, f"{bad} infants with dangling pregnancy_id"
    print("synth_infant:", spark.table(f"{FINAL}.synth_infant").count())


# ---------------------------------------------------------------------------
# 4. infant visits + visit_detail (designate nnu Inpatient parent; 1 detail per nnu visit)
# ---------------------------------------------------------------------------
def _infant_visits_and_detail():
    vd_target = TARGET["visit_detail"]
    n_inf_visits = spark.sql(
        f"SELECT COUNT(*) c FROM {FINAL}.synth_visit_occurrence v "
        f"JOIN {FINAL}.synth_person p ON p.person_id=v.person_id WHERE p.person_role='infant'"
    ).first()["c"]
    if n_inf_visits == 0:
        spark.sql(f"CREATE OR REPLACE TABLE {FINAL}.synth_visit_detail AS "
                  f"SELECT * FROM {SRC}.visit_detail WHERE 1=0")
        print("synth_visit_detail: 0 (no infant visits)")
        return
    frac_bps = int(round(min(1.0, vd_target / n_inf_visits) * 1_000_000))
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._nnu_visits AS
      SELECT v.visit_occurrence_id, v.person_id, v.pregnancy_id,
             v.visit_start_date, v.visit_start_datetime, v.visit_end_date, v.visit_end_datetime
      FROM {FINAL}.synth_visit_occurrence v
      JOIN {FINAL}.synth_person p ON p.person_id=v.person_id
      WHERE p.person_role='infant'
        AND pmod(abs(hash(v.visit_occurrence_id)), 1000000) < {frac_bps}""")
    spark.sql(f"""MERGE INTO {FINAL}.synth_visit_occurrence v
      USING {FINAL}._nnu_visits n ON v.visit_occurrence_id=n.visit_occurrence_id
      WHEN MATCHED THEN UPDATE SET v.visit_concept_id={VISIT_INPATIENT}""")

    vdcols = _full_cols("visit_detail")
    vd_id = M("enrich-nnu-visitdetail", "n.visit_occurrence_id")
    defaults = {
        "visit_detail_id": f"{vd_id} AS visit_detail_id",
        "person_id": "n.person_id",
        "visit_detail_concept_id": f"{VISIT_DETAIL_CONCEPT} AS visit_detail_concept_id",
        "visit_detail_type_concept_id": f"{VISIT_DETAIL_TYPE} AS visit_detail_type_concept_id",
        "visit_detail_start_date": "n.visit_start_date AS visit_detail_start_date",
        "visit_detail_start_datetime": "n.visit_start_datetime AS visit_detail_start_datetime",
        "visit_detail_end_date": "n.visit_end_date AS visit_detail_end_date",
        "visit_detail_end_datetime": "n.visit_end_datetime AS visit_detail_end_datetime",
        "visit_occurrence_id": "n.visit_occurrence_id AS visit_occurrence_id",
        "pregnancy_id": "n.pregnancy_id AS pregnancy_id",
        "site_id": f"'{SITE_ID}' AS site_id",
        "source_table": "'nnu' AS source_table",
        "enrichment_tag": "'nnu' AS enrichment_tag",
    }
    parts = [defaults.get(c, f"CAST(NULL AS {_sqltype('visit_detail', c)}) AS `{c}`") for c in vdcols]
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_visit_detail AS
      SELECT {', '.join(parts)} FROM {FINAL}._nnu_visits n""")
    orphan = spark.sql(f"""SELECT COUNT(*) c FROM {FINAL}.synth_visit_detail vd
      WHERE NOT EXISTS (SELECT 1 FROM {FINAL}.synth_visit_occurrence vo
        WHERE vo.visit_occurrence_id=vd.visit_occurrence_id)""").first()["c"]
    assert orphan == 0, f"{orphan} visit_detail rows without a parent visit"
    print("synth_visit_detail:", spark.table(f"{FINAL}.synth_visit_detail").count(),
          "| parent resolves for all")


# ---------------------------------------------------------------------------
# 5. link: visit_occurrence -> pregnancy FIRST, then facts -> visit (inherit pregnancy)
# ---------------------------------------------------------------------------
def _link():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._link_visit AS
      WITH mother AS (
        SELECT v.visit_occurrence_id, pr.pregnancy_id,
               ROW_NUMBER() OVER (PARTITION BY v.visit_occurrence_id
                 ORDER BY ABS(DATEDIFF(v.visit_start_date, pr.pregnancy_start_date)) ASC,
                          pr.pregnancy_id ASC) AS rn
        FROM {FINAL}.synth_visit_occurrence v
        JOIN {FINAL}.synth_person p ON p.person_id=v.person_id AND p.person_role='mother'
        JOIN {FINAL}.synth_pregnancy pr ON pr.person_id=v.person_id
      ),
      infant AS (
        SELECT v.visit_occurrence_id, m.pregnancy_id, 1 AS rn
        FROM {FINAL}.synth_visit_occurrence v
        JOIN {FINAL}.synth_person p ON p.person_id=v.person_id AND p.person_role='infant'
        JOIN {FINAL}._infant_map m ON m.infant_person_id=v.person_id
      )
      SELECT visit_occurrence_id, pregnancy_id FROM mother WHERE rn=1
      UNION ALL SELECT visit_occurrence_id, pregnancy_id FROM infant""")
    spark.sql(f"""MERGE INTO {FINAL}.synth_visit_occurrence v
      USING {FINAL}._link_visit l ON v.visit_occurrence_id=l.visit_occurrence_id
      WHEN MATCHED THEN UPDATE SET v.pregnancy_id=l.pregnancy_id""")
    spark.sql(f"DROP TABLE IF EXISTS {FINAL}._link_visit")
    print("linked visit_occurrence -> pregnancy")

    date_of = {"measurement": "measurement_date", "observation": "observation_date",
               "condition_occurrence": "condition_start_date",
               "procedure_occurrence": "procedure_date"}
    for t, date_col in date_of.items():
        pk = f"{t}_id"
        spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._link_{t} AS
          WITH cand AS (
            SELECT f.{pk} AS fid, v.visit_occurrence_id, v.pregnancy_id,
                   ROW_NUMBER() OVER (PARTITION BY f.{pk}
                     ORDER BY ABS(DATEDIFF(f.`{date_col}`, v.visit_start_date)) ASC,
                              v.visit_occurrence_id ASC) AS rn
            FROM {FINAL}.synth_{t} f
            JOIN {FINAL}.synth_visit_occurrence v ON v.person_id=f.person_id
          )
          SELECT fid, visit_occurrence_id, pregnancy_id FROM cand WHERE rn=1""")
        spark.sql(f"""MERGE INTO {FINAL}.synth_{t} f
          USING {FINAL}._link_{t} l ON f.{pk}=l.fid
          WHEN MATCHED THEN UPDATE SET
            f.visit_occurrence_id=l.visit_occurrence_id, f.pregnancy_id=l.pregnancy_id""")
        spark.sql(f"DROP TABLE IF EXISTS {FINAL}._link_{t}")
        print(f"linked {t} -> visit + pregnancy")


# ---------------------------------------------------------------------------
# 6. episode (1/pregnancy) + episode_event (5 fact tables via pregnancy_id)
# ---------------------------------------------------------------------------
def _episode_and_events():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_episode AS
      SELECT {M('episode','pr.pregnancy_id')} AS episode_id,
        pr.person_id, {EPISODE_CONCEPT} AS episode_concept_id,
        pr.pregnancy_start_date AS episode_start_date,
        CAST(pr.pregnancy_start_date AS TIMESTAMP) AS episode_start_datetime,
        CASE WHEN pr.pregnancy_end_date < pr.pregnancy_start_date THEN NULL
             ELSE pr.pregnancy_end_date END AS episode_end_date,
        CAST(CASE WHEN pr.pregnancy_end_date < pr.pregnancy_start_date THEN NULL
             ELSE pr.pregnancy_end_date END AS TIMESTAMP) AS episode_end_datetime,
        CAST(NULL AS BIGINT) AS episode_parent_id,
        CAST(NULL AS INT) AS episode_number,
        {EPISODE_OBJECT_PREGNANCY} AS episode_object_concept_id,
        {EPISODE_TYPE} AS episode_type_concept_id,
        CAST(NULL AS STRING) AS episode_source_value,
        CAST(NULL AS BIGINT) AS episode_source_concept_id,
        pr.pregnancy_id, '{SITE_ID}' AS site_id
      FROM {FINAL}.synth_pregnancy pr""")
    en = spark.table(f"{FINAL}.synth_episode").count()
    ed = spark.sql(f"SELECT COUNT(DISTINCT episode_id) c FROM {FINAL}.synth_episode").first()["c"]
    pn = spark.table(f"{FINAL}.synth_pregnancy").count()
    assert en == ed, f"episode_id collision {en} vs {ed}"
    assert en == pn, f"episode not 1:1 with pregnancy ({en} vs {pn})"
    print("synth_episode:", en, "(1 per pregnancy)")

    links = []
    for src_tbl, (fc, idcol) in EPISODE_EVENT_FIELDS.items():
        links.append(f"""
          SELECT e.episode_id, f.{idcol} AS event_id, {fc} AS episode_event_field_concept_id,
                 '{src_tbl}' AS source_table, '{SITE_ID}' AS site_id
          FROM {FINAL}.synth_{src_tbl} f
          JOIN {FINAL}.synth_episode e ON e.pregnancy_id = f.pregnancy_id
          WHERE f.pregnancy_id IS NOT NULL""")
    spark.sql(f"CREATE OR REPLACE TABLE {FINAL}.synth_episode_event AS "
              + "\nUNION ALL\n".join(links))
    print("synth_episode_event:", spark.table(f"{FINAL}.synth_episode_event").count())


# ---------------------------------------------------------------------------
# 7. observation_period (one per person, span of events) — ported _common.py:2389
# ---------------------------------------------------------------------------
def _observation_period():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_observation_period AS
      WITH ev AS (
        SELECT person_id, condition_start_date AS d FROM {FINAL}.synth_condition_occurrence
        UNION ALL SELECT person_id, measurement_date FROM {FINAL}.synth_measurement
        UNION ALL SELECT person_id, observation_date FROM {FINAL}.synth_observation
        UNION ALL SELECT person_id, procedure_date FROM {FINAL}.synth_procedure_occurrence
        UNION ALL SELECT person_id, visit_start_date FROM {FINAL}.synth_visit_occurrence
        UNION ALL SELECT person_id, visit_end_date FROM {FINAL}.synth_visit_occurrence
        UNION ALL SELECT person_id, episode_start_date FROM {FINAL}.synth_episode
        UNION ALL SELECT person_id, episode_end_date FROM {FINAL}.synth_episode
      ),
      agg AS (SELECT person_id, min(d) mn, max(d) mx FROM ev WHERE d IS NOT NULL GROUP BY person_id),
      pf AS (SELECT person_id, min(pregnancy_start_date) ps,
                    max(coalesce(pregnancy_end_date, pregnancy_start_date)) pe
             FROM {FINAL}.synth_pregnancy GROUP BY person_id)
      SELECT {M('obs-period','p.person_id')} AS observation_period_id, p.person_id,
        coalesce(a.mn, pf.ps, CAST(p.birth_datetime AS DATE)) AS observation_period_start_date,
        coalesce(a.mx, pf.pe, CAST(p.birth_datetime AS DATE)) AS observation_period_end_date,
        {OBS_PERIOD_TYPE} AS period_type_concept_id, '{SITE_ID}' AS site_id
      FROM {FINAL}.synth_person p
      LEFT JOIN agg a ON a.person_id=p.person_id
      LEFT JOIN pf ON pf.person_id=p.person_id""")
    n = spark.table(f"{FINAL}.synth_observation_period").count()
    pc = spark.table(f"{FINAL}.synth_person").count()
    nn = spark.sql(f"""SELECT COUNT(*) c FROM {FINAL}.synth_observation_period
      WHERE observation_period_start_date IS NULL OR observation_period_end_date IS NULL""").first()["c"]
    assert n == pc, f"observation_period not 1:1 with person ({n} vs {pc})"
    assert nn == 0, f"{nn} observation_period rows with null dates"
    print("synth_observation_period:", n, "(1 per person, no null dates)")


# ---------------------------------------------------------------------------
# 8. fact_relationship (mother/child/twin/sibling) — ported _common.py:1023
# ---------------------------------------------------------------------------
def _fact_relationship():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}._mc_base AS
      SELECT i.pregnancy_id, pr.person_id AS mother_id, i.infant_id AS child_id
      FROM {FINAL}.synth_infant i
      JOIN {FINAL}.synth_pregnancy pr ON pr.pregnancy_id=i.pregnancy_id
      WHERE i.pregnancy_id IS NOT NULL""")
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_fact_relationship AS
      SELECT {PERSON_DOMAIN} AS domain_concept_id_1, mother_id AS fact_id_1,
             {PERSON_DOMAIN} AS domain_concept_id_2, child_id AS fact_id_2,
             {REL_MOTHER} AS relationship_concept_id, '{SITE_ID}' AS site_id FROM {FINAL}._mc_base
      UNION ALL
      SELECT {PERSON_DOMAIN}, child_id, {PERSON_DOMAIN}, mother_id, {REL_CHILD}, '{SITE_ID}' FROM {FINAL}._mc_base
      UNION ALL
      SELECT {PERSON_DOMAIN}, a.child_id, {PERSON_DOMAIN}, b.child_id, {REL_TWIN}, '{SITE_ID}'
        FROM {FINAL}._mc_base a JOIN {FINAL}._mc_base b
          ON a.pregnancy_id=b.pregnancy_id AND a.child_id<>b.child_id
      UNION ALL
      SELECT {PERSON_DOMAIN}, a.child_id, {PERSON_DOMAIN}, b.child_id, {REL_SIBLING}, '{SITE_ID}'
        FROM {FINAL}._mc_base a JOIN {FINAL}._mc_base b
          ON a.mother_id=b.mother_id AND a.pregnancy_id<>b.pregnancy_id AND a.child_id<>b.child_id""")
    spark.sql(f"DROP TABLE IF EXISTS {FINAL}._mc_base")
    by = spark.sql(f"SELECT relationship_concept_id, COUNT(*) c FROM {FINAL}.synth_fact_relationship "
                   f"GROUP BY 1 ORDER BY 1").collect()
    print("synth_fact_relationship:", spark.table(f"{FINAL}.synth_fact_relationship").count(),
          "| by concept", [(r["relationship_concept_id"], r["c"]) for r in by])


# ---------------------------------------------------------------------------
# 9. cdm_source — describe the SYNTHETIC release (not copied)
# ---------------------------------------------------------------------------
def _cdm_source():
    spark.sql(f"""CREATE OR REPLACE TABLE {FINAL}.synth_cdm_source AS
      SELECT
        'PREBORN maternity OMOP subset (SYNTHETIC)' AS cdm_source_name,
        'PREBORN-SYNTH' AS cdm_source_abbreviation,
        'Barts Health NHS Trust' AS cdm_holder,
        'DP-synthetic copy (MOSTLY AI) of the PREBORN maternity OMOP CDM v5.4 subset' AS source_description,
        CAST(NULL AS STRING) AS source_documentation_reference,
        CAST(NULL AS STRING) AS cdm_etl_reference,
        DATE'{RELEASE_DATE}' AS source_release_date,
        DATE'{RELEASE_DATE}' AS cdm_release_date,
        'v5.4' AS cdm_version,
        CAST({CDM_VERSION_CONCEPT} AS BIGINT) AS cdm_version_concept_id,
        (SELECT vocabulary_version FROM {VOCAB_SRC}.vocabulary WHERE vocabulary_id='None') AS vocabulary_version""")
    print("synth_cdm_source written (synthetic release)")


# ---------------------------------------------------------------------------
# 10. used-vocabulary (bundled, pruned to synthetic used-set) — ported _common.py:2519
# ---------------------------------------------------------------------------
def build_used_vocabulary_preborn(schema):
    S = schema
    P = VOCAB_SRC
    pfx = SYNTH_PREFIX
    COLS = {
      "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
                 "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
      "observation_period": ["period_type_concept_id"],
      "condition_occurrence": ["condition_concept_id", "condition_type_concept_id",
                               "condition_status_concept_id", "condition_source_concept_id"],
      "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id",
                               "modifier_concept_id", "procedure_source_concept_id"],
      "measurement": ["measurement_concept_id", "measurement_type_concept_id", "operator_concept_id",
                      "value_as_concept_id", "unit_concept_id", "measurement_source_concept_id",
                      "unit_source_concept_id", "meas_event_field_concept_id"],
      "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                      "qualifier_concept_id", "unit_concept_id", "observation_source_concept_id",
                      "obs_event_field_concept_id"],
      "visit_occurrence": ["visit_concept_id", "visit_type_concept_id", "visit_source_concept_id",
                           "admitted_from_concept_id", "discharged_to_concept_id"],
      "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id",
                       "visit_detail_source_concept_id", "admitted_from_concept_id",
                       "discharged_to_concept_id"],
      "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id",
                  "episode_source_concept_id"],
      "episode_event": ["episode_event_field_concept_id"],
      "fact_relationship": ["domain_concept_id_1", "domain_concept_id_2", "relationship_concept_id"],
      "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                    "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                    "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
      "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
      "cdm_source": ["cdm_version_concept_id"],
    }
    sel = []
    for t, cols in COLS.items():
        for c in cols:
            sel.append(f"SELECT CAST({c} AS BIGINT) AS cid FROM {S}.{pfx}{t}")
    union_sql = "\nUNION ALL\n".join(sel)
    spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_used AS
      SELECT DISTINCT cid FROM ({union_sql}) WHERE cid IS NOT NULL AND cid>0""")
    used_n = spark.table(f"{S}._vocab_used").count()

    spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_ids AS
      SELECT cid FROM {S}._vocab_used
      UNION SELECT CAST(ca.ancestor_concept_id AS BIGINT) FROM {P}.concept_ancestor ca
        JOIN {S}._vocab_used u ON ca.descendant_concept_id=u.cid
      UNION SELECT CAST(vocabulary_concept_id AS BIGINT) FROM {P}.vocabulary WHERE vocabulary_concept_id>0
      UNION SELECT CAST(domain_concept_id AS BIGINT) FROM {P}.domain WHERE domain_concept_id>0
      UNION SELECT CAST(concept_class_concept_id AS BIGINT) FROM {P}.concept_class WHERE concept_class_concept_id>0
      UNION SELECT CAST(relationship_concept_id AS BIGINT) FROM {P}.relationship WHERE relationship_concept_id>0""")
    for _ in range(5):
        prev = spark.table(f"{S}._vocab_ids").count()
        spark.sql(f"""CREATE OR REPLACE TABLE {S}._vocab_ids_next AS
          SELECT cid FROM {S}._vocab_ids
          UNION SELECT CAST(cs.language_concept_id AS BIGINT) FROM {P}.concept_synonym cs
            JOIN {S}._vocab_ids v ON cs.concept_id=v.cid
            WHERE cs.language_concept_id IS NOT NULL AND cs.language_concept_id>0""")
        spark.sql(f"CREATE OR REPLACE TABLE {S}._vocab_ids AS SELECT cid FROM {S}._vocab_ids_next")
        if spark.table(f"{S}._vocab_ids").count() == prev:
            break
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_ids_next")

    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}concept AS
      SELECT CAST(c.concept_id AS BIGINT) AS concept_id, c.concept_name, c.domain_id, c.vocabulary_id,
             c.concept_class_id, c.standard_concept, c.concept_code,
             c.valid_start_date, c.valid_end_date, c.invalid_reason
      FROM {P}.concept c JOIN {S}._vocab_ids v ON c.concept_id=v.cid""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}concept_relationship AS
      SELECT CAST(cr.concept_id_1 AS BIGINT) concept_id_1, CAST(cr.concept_id_2 AS BIGINT) concept_id_2,
             cr.relationship_id, cr.valid_start_date, cr.valid_end_date, cr.invalid_reason
      FROM {P}.concept_relationship cr
      WHERE cr.concept_id_1 IN (SELECT concept_id FROM {S}.{pfx}concept)
        AND cr.concept_id_2 IN (SELECT concept_id FROM {S}.{pfx}concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}concept_ancestor AS
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) ancestor_concept_id,
             CAST(ca.descendant_concept_id AS BIGINT) descendant_concept_id,
             CAST(ca.min_levels_of_separation AS BIGINT) min_levels_of_separation,
             CAST(ca.max_levels_of_separation AS BIGINT) max_levels_of_separation
      FROM {P}.concept_ancestor ca
      WHERE ca.descendant_concept_id IN (SELECT cid FROM {S}._vocab_used)
        AND ca.ancestor_concept_id IN (SELECT concept_id FROM {S}.{pfx}concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}concept_synonym AS
      SELECT CAST(cs.concept_id AS BIGINT) concept_id, cs.concept_synonym_name,
             CAST(cs.language_concept_id AS BIGINT) language_concept_id
      FROM {P}.concept_synonym cs WHERE cs.concept_id IN (SELECT concept_id FROM {S}.{pfx}concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}vocabulary AS
      SELECT vocabulary_id, vocabulary_name, vocabulary_reference, vocabulary_version,
             CAST(vocabulary_concept_id AS BIGINT) vocabulary_concept_id FROM {P}.vocabulary""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}domain AS
      SELECT domain_id, domain_name, CAST(domain_concept_id AS BIGINT) domain_concept_id FROM {P}.domain""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}concept_class AS
      SELECT concept_class_id, concept_class_name,
             CAST(concept_class_concept_id AS BIGINT) concept_class_concept_id FROM {P}.concept_class""")
    spark.sql(f"""CREATE OR REPLACE TABLE {S}.{pfx}relationship AS
      SELECT relationship_id, relationship_name, is_hierarchical, defines_ancestry,
             reverse_relationship_id, CAST(relationship_concept_id AS BIGINT) relationship_concept_id
      FROM {P}.relationship""")
    concept_n = spark.table(f"{S}.{pfx}concept").count()

    checks = [
      ("cr.c1", f"SELECT count(*) c FROM {S}.{pfx}concept_relationship r LEFT JOIN {S}.{pfx}concept c ON c.concept_id=r.concept_id_1 WHERE c.concept_id IS NULL"),
      ("cr.c2", f"SELECT count(*) c FROM {S}.{pfx}concept_relationship r LEFT JOIN {S}.{pfx}concept c ON c.concept_id=r.concept_id_2 WHERE c.concept_id IS NULL"),
      ("ca.anc", f"SELECT count(*) c FROM {S}.{pfx}concept_ancestor a LEFT JOIN {S}.{pfx}concept c ON c.concept_id=a.ancestor_concept_id WHERE c.concept_id IS NULL"),
      ("ca.desc", f"SELECT count(*) c FROM {S}.{pfx}concept_ancestor a LEFT JOIN {S}.{pfx}concept c ON c.concept_id=a.descendant_concept_id WHERE c.concept_id IS NULL"),
      ("cs.cid", f"SELECT count(*) c FROM {S}.{pfx}concept_synonym s LEFT JOIN {S}.{pfx}concept c ON c.concept_id=s.concept_id WHERE c.concept_id IS NULL"),
      ("concept.domain", f"SELECT count(*) c FROM {S}.{pfx}concept c LEFT JOIN {S}.{pfx}domain d ON d.domain_id=c.domain_id WHERE d.domain_id IS NULL"),
      ("concept.vocab", f"SELECT count(*) c FROM {S}.{pfx}concept c LEFT JOIN {S}.{pfx}vocabulary v ON v.vocabulary_id=c.vocabulary_id WHERE v.vocabulary_id IS NULL"),
      ("concept.class", f"SELECT count(*) c FROM {S}.{pfx}concept c LEFT JOIN {S}.{pfx}concept_class k ON k.concept_class_id=c.concept_class_id WHERE k.concept_class_id IS NULL"),
    ]
    dangling = 0
    print(f"build_used_vocabulary_preborn: used={used_n} concept_rows={concept_n}")
    for name, qy in checks:
        n = spark.sql(qy).first()["c"]; dangling += n
        print(f"  FK-check [{'OK ' if n==0 else 'FAIL'}] {name}: {n}")
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_used")
    spark.sql(f"DROP TABLE IF EXISTS {S}._vocab_ids")
    assert dangling == 0, f"{dangling} dangling vocab FK(s)"
    print("build_used_vocabulary_preborn DONE: 0-dangling")


# ---------------------------------------------------------------------------
# 11. cleanup — drop working helper tables from the final schema
# ---------------------------------------------------------------------------
def _cleanup_helpers():
    for h in ["_infant_map", "_preg_slots", "_nnu_visits"]:
        spark.sql(f"DROP TABLE IF EXISTS {FINAL}.{h}")
    print("helper tables dropped (_infant_map, _preg_slots, _nnu_visits)")


def _summary():
    print("\n=== synth_* row counts ===")
    for t in ["person", "observation_period", "pregnancy", "infant", "visit_occurrence",
              "visit_detail", "measurement", "observation", "condition_occurrence",
              "procedure_occurrence", "episode", "episode_event", "fact_relationship",
              "cdm_source", "concept"]:
        try:
            print(f"  synth_{t}: {spark.table(f'{FINAL}.synth_{t}').count():,}")
        except Exception as e:
            print(f"  synth_{t}: (missing) {e}")


print("preborn_derive loaded: derive_all() runs Phase B on the staged synthetic base")